# Permutation importance and TreeSHAP

Run from top to bottom. Quick mode is a patient subset, not final results. Notebook outputs are separate from production reports.

In [ ]:
from pathlib import Path
import sys, os
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src'/'features.py').exists()), None)
assert ROOT is not None, 'Open the notebook from the project repository'
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.notebook_utils import *
QUICK = True  # False uses the entire eligible dataset; keep consistent across notebooks
OUT = ROOT/'reports'/('notebook_quick' if QUICK else 'notebook_full')
OUT.mkdir(parents=True, exist_ok=True)


Fit the model on training data

In [ ]:
raw, x, y, g, parts = load_cohort(ROOT, QUICK)
train, cal, tune, test = (parts[k] for k in ("train","calibration","threshold","test"))
X, Y, G = x.iloc[train], y.iloc[train], g.iloc[train]
display(split_table(y,g,parts))

Calculate feature importance on the validation partition

In [ ]:
from src.models.boosting import models
from src.evaluation.explainability import explain
m=make_model(X,models()['XGBoost'],quick=QUICK).fit(X,Y)
explain(m,X,x.iloc[tune],y.iloc[tune],OUT,m)

Plot actual computed importances. TreeSHAP contributions refer to the XGBoost margin, not calibrated probabilities.

In [ ]:
for file,col,label in [('permutation_importance.csv','importance','feature'),('xgboost_shap.csv','mean_absolute_shap','encoded_feature')]:
    d=pd.read_csv(OUT/file).head(15); display(d); d.set_index(label)[col].sort_values().plot.barh(figsize=(9,5)); plt.tight_layout(); plt.show()